# Scaling Experiments
Trains both IBM Model 1 and the phrase-based SMT system on 50K, 100K, 250K, and 500K
Europarl sentence pairs and evaluates each on the Koehn-style in-domain held-out test set
(1,755 sentences with French source length 5–15 tokens).  Results go into Table 1 of the
writeup and the scaling-analysis paragraph in §4.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
!pip install -q sacrebleu

# Compile fast_align (needed for phrase-based alignments).
!apt-get -q install -y cmake > /dev/null 2>&1
!git clone -q https://github.com/clab/fast_align.git
!cd fast_align && mkdir -p build && cd build && cmake -DCMAKE_BUILD_TYPE=Release .. -DCMAKE_VERBOSE_MAKEFILE=OFF > /dev/null 2>&1 && make -j2 > /dev/null 2>&1
import os
os.environ['FAST_ALIGN'] = 'fast_align/build/fast_align'
os.environ['ATOOLS']     = 'fast_align/build/atools'
print('fast_align compiled:', os.path.exists('fast_align/build/fast_align'))

In [ ]:
# ── Download Europarl ─────────────────────────────────────────────────────────
!wget -q http://www.statmt.org/europarl/v7/fr-en.tgz
!tar -xzf fr-en.tgz
print('Europarl downloaded.')

In [ ]:
# ── Load corpus and define fixed test split ───────────────────────────────────
from collections import defaultdict, Counter
import math, os, sacrebleu, subprocess, time

def tokenize(s):
    return s.lower().strip().split()

with open('europarl-v7.fr-en.fr', encoding='utf-8') as f:
    fr_raw = [l.strip() for l in f]
with open('europarl-v7.fr-en.en', encoding='utf-8') as f:
    en_raw = [l.strip() for l in f]

pairs  = [(f, e) for f, e in zip(fr_raw, en_raw) if f and e]
fr_raw = [p[0] for p in pairs]
en_raw = [p[1] for p in pairs]
n = len(fr_raw)
print(f'Total sentence pairs: {n:,}')

# Fixed Koehn-style test split — same for all training sizes.
EUROPARL_TEST_SIZE = 1755
TEST_SRC_MIN_LEN   = 5
TEST_SRC_MAX_LEN   = 15

qual_idx     = [i for i in range(n) if TEST_SRC_MIN_LEN <= len(tokenize(fr_raw[i])) <= TEST_SRC_MAX_LEN]
test_idx_set = set(qual_idx[-EUROPARL_TEST_SIZE:])
test_order   = sorted(test_idx_set)
test_fr_tok  = [tokenize(fr_raw[i]) for i in test_order]
test_fr_raw  = [fr_raw[i]           for i in test_order]
test_en_ref  = [en_raw[i]           for i in test_order]
print(f'Fixed test set: {len(test_order):,} sentences')

In [ ]:
# ── IBM Model 1 training + evaluation helper ──────────────────────────────────

def train_ibm1(train_fr, train_en, iterations=10):
    fr_to_en = defaultdict(set)
    for fr_s, en_s in zip(train_fr, train_en):
        for f in fr_s:
            for e in en_s:
                fr_to_en[f].add(e)
    t = defaultdict(float)
    for f, en_words in fr_to_en.items():
        for e in en_words:
            t[(e, f)] = 1.0 / len(en_words)
    for _ in range(iterations):
        count = defaultdict(float)
        total = defaultdict(float)
        for en_s, fr_s in zip(train_en, train_fr):
            for e in en_s:
                s = sum(t[(e, f)] for f in fr_s)
                if s == 0:
                    continue
                for f in fr_s:
                    d = t[(e, f)] / s
                    count[(e, f)] += d
                    total[f]      += d
        for (e, f) in count:
            if total[f]:
                t[(e, f)] = count[(e, f)] / total[f]
    best = {}
    for (e, f), p in t.items():
        if f not in best or p > best[f][1]:
            best[f] = (e, p)
    return best

def ibm_translate(best, tokens):
    return ' '.join(best[w][0] if w in best else '<UNK>' for w in tokens)

def ibm_bleu(best, test_fr_tok, test_en_ref):
    hyps = [ibm_translate(best, s) for s in test_fr_tok]
    return sacrebleu.corpus_bleu(hyps, [test_en_ref], lowercase=True).score

In [ ]:
# ── Phrase-based SMT pipeline (from Phrase_model_new.ipynb) ───────────────────
NULL_TOKEN = '<NULL>'

def parse_pharaoh(line):
    links = set()
    for pair in line.strip().split():
        if '-' in pair:
            a, b = pair.split('-', 1)
            if a.isdigit() and b.isdigit():
                links.add((int(a), int(b)))
    return links

def get_phrases(fr_s, en_s, links, max_len=3):
    lf, le = len(fr_s), len(en_s)
    if not lf or not le:
        return []
    by_e = defaultdict(set)
    by_f = defaultdict(set)
    for i, j in links:
        if i < lf and j < le:
            by_e[j].add(i); by_f[i].add(j)
    phrases = []
    for i1 in range(lf):
        js = set()
        for i2 in range(i1, min(i1+max_len, lf)):
            js.update(by_f.get(i2, set()))
            if not js:
                continue
            j1, j2 = min(js), max(js)
            if j2-j1+1 > max_len:
                continue
            ok = True
            for j in range(j1, j2+1):
                for ia in by_e.get(j, set()):
                    if ia < i1 or ia > i2:
                        ok = False; break
                if not ok:
                    break
            if not ok:
                continue
            loc = tuple((i-i1, j-j1) for i, j in links if i1<=i<=i2 and j1<=j<=j2)
            phrases.append((tuple(fr_s[i1:i2+1]), tuple(en_s[j1:j2+1]), loc))
    return phrases

def build_lex_tables(fr_corp, en_corp, aligns):
    cef = defaultdict(Counter)
    cfe = defaultdict(Counter)
    for fr_s, en_s, links in zip(fr_corp, en_corp, aligns):
        lf, le = len(fr_s), len(en_s)
        af, ae = set(), set()
        for i, j in links:
            if i < lf and j < le:
                cef[fr_s[i]][en_s[j]] += 1
                cfe[en_s[j]][fr_s[i]] += 1
                af.add(i); ae.add(j)
        for j in range(le):
            if j not in ae: cef[NULL_TOKEN][en_s[j]] += 1
        for i in range(lf):
            if i not in af: cfe[NULL_TOKEN][fr_s[i]] += 1
    pw_ef = {f: {e: c/sum(ctr.values()) for e, c in ctr.items()} for f, ctr in cef.items()}
    pw_fe = {e: {f: c/sum(ctr.values()) for f, c in ctr.items()} for e, ctr in cfe.items()}
    return pw_ef, pw_fe

def lex_w(phrase_src, phrase_tgt, loc, pw_st):
    al = defaultdict(list)
    for i, j in loc: al[j].append(i)
    prod = 1.0
    for j, tw in enumerate(phrase_tgt):
        srcs = al.get(j)
        if srcs:
            prod *= sum(pw_st.get(phrase_src[i], {}).get(tw, 1e-9) for i in srcs) / len(srcs)
        else:
            prod *= pw_st.get(NULL_TOKEN, {}).get(tw, 1e-9)
    return prod

def build_phrase_table(fr_corp, en_corp, aligns, pw_ef, pw_fe, max_len=3, top_k=20):
    pc = Counter(); fc = Counter(); ec = Counter()
    best_ef = {}; best_fe = {}
    for fr_s, en_s, links in zip(fr_corp, en_corp, aligns):
        for fp, ep, loc in get_phrases(fr_s, en_s, links, max_len):
            key = (fp, ep)
            pc[key] += 1; fc[fp] += 1; ec[ep] += 1
            lef = lex_w(fp, ep, loc, pw_ef)
            lfe = lex_w(ep, fp, [(j,i) for i,j in loc], pw_fe)
            if key not in best_ef or lef > best_ef[key]: best_ef[key] = lef
            if key not in best_fe or lfe > best_fe[key]: best_fe[key] = lfe
    table = defaultdict(list)
    for (fp, ep), c in pc.items():
        table[fp].append({
            'en':        ep,
            'log_phi_ef': math.log(c / fc[fp]),
            'log_phi_fe': math.log(c / ec[ep]),
            'log_lex_ef': math.log(max(best_ef[(fp,ep)], 1e-20)),
            'log_lex_fe': math.log(max(best_fe[(fp,ep)], 1e-20)),
        })
    for fp in table:
        table[fp].sort(key=lambda o: o['log_phi_ef'], reverse=True)
        table[fp] = table[fp][:top_k]
    return dict(table)

def build_lm(en_corp):
    uni = Counter(); bi = Counter(); tri = Counter()
    bi_ctx = Counter(); uni_ctx = Counter(); total = 0
    for s in en_corp:
        seq = ['<s>','<s>'] + list(s) + ['</s>']
        for w in seq: uni[w] += 1; total += 1
        for i in range(1, len(seq)):
            bi[(seq[i-1], seq[i])] += 1; uni_ctx[seq[i-1]] += 1
        for i in range(2, len(seq)):
            tri[(seq[i-2], seq[i-1], seq[i])] += 1; bi_ctx[(seq[i-2], seq[i-1])] += 1
    la = math.log(0.4); lf = math.log(1/(total+len(uni)))
    def lp(w, c1, c2):
        if tri.get((c1,c2,w)): return math.log(tri[(c1,c2,w)] / bi_ctx[(c1,c2)])
        if bi.get((c2,w)) and uni_ctx.get(c2): return la + math.log(bi[(c2,w)] / uni_ctx[c2])
        if uni.get(w): return 2*la + math.log(uni[w]/total)
        return 2*la + lf
    return lp

W = {'log_phi_ef':1.0,'log_phi_fe':0.5,'log_lex_ef':0.5,'log_lex_fe':0.5,
     'lm':1.0,'word_bonus':0.5,'phrase_penalty':0.0,'distortion':-1.0}
DIST_LIMIT = 2; BEAM = 32; MAX_OPTS = 4; MAX_PH = 3

def static_score(opt, W):
    return (W['log_phi_ef']*opt['log_phi_ef'] + W['log_phi_fe']*opt['log_phi_fe'] +
            W['log_lex_ef']*opt['log_lex_ef'] + W['log_lex_fe']*opt['log_lex_fe'] +
            W['word_bonus']*len(opt['en'])    + W['phrase_penalty'])

def build_options(fr_s, ptable, fallback, pw_ef, max_ph, max_opts):
    N = len(fr_s); opts = {}
    for i in range(N):
        for j in range(i+1, min(i+max_ph, N)+1):
            fp = tuple(fr_s[i:j])
            if fp in ptable:
                opts[(i,j)] = ptable[fp][:max_opts]
            elif j == i+1:
                w = fp[0]; alts = pw_ef.get(w)
                cands = []
                if alts:
                    for e, p in sorted(alts.items(), key=lambda x: x[1], reverse=True)[:3]:
                        if e != NULL_TOKEN:
                            lp = math.log(max(p, 1e-12))
                            cands.append({'en':(e,),'log_phi_ef':lp,'log_phi_fe':lp,'log_lex_ef':lp,'log_lex_fe':lp})
                if not cands:
                    fb = fallback.get(w, w); lp = math.log(1e-5)
                    cands.append({'en':(fb,),'log_phi_ef':lp,'log_phi_fe':lp,'log_lex_ef':lp,'log_lex_fe':lp})
                opts[(i,j)] = cands
    return opts

def build_fc(opts, N, lm, W):
    NEG = float('-inf')
    fc = [[NEG]*(N+1) for _ in range(N+1)]
    for (i,j), os_ in opts.items():
        best = NEG
        for o in os_:
            lm_s = 0.0; ctx = ('<s>','<s>')
            for w in o['en']: lm_s += lm(w, ctx[0], ctx[1]); ctx = (ctx[1], w)
            cand = static_score(o, W) + W['lm']*lm_s
            if cand > best: best = cand
        fc[i][j] = best
    for length in range(2, N+1):
        for i in range(N-length+1):
            j = i+length; best = fc[i][j]
            for k in range(i+1, j):
                c = fc[i][k]+fc[k][j]
                if c > best: best = c
            fc[i][j] = best
    return fc

def fc_mask(mask, N, fc, cache):
    if mask in cache: return cache[mask]
    tot = 0.0; i = 0
    while i < N:
        if (mask>>i)&1: i += 1; continue
        j = i
        while j < N and not ((mask>>j)&1): j += 1
        tot += fc[i][j]; i = j
    cache[mask] = tot; return tot

def phrase_decode(fr_s, ptable, fallback, pw_ef, lm, W,
                  max_ph=MAX_PH, dlimit=DIST_LIMIT, beam=BEAM, max_opts=MAX_OPTS):
    N = len(fr_s)
    if N == 0: return []
    opts = build_options(fr_s, ptable, fallback, pw_ef, max_ph, max_opts)
    fc   = build_fc(opts, N, lm, W)
    sm   = {(i,j): ((1<<(j-i))-1)<<i for (i,j) in opts}
    full = (1<<N)-1; fcc = {full: 0.0}
    stacks = [dict() for _ in range(N+1)]
    stacks[0][(0,-1,('<s>','<s>'))] = (fc_mask(0,N,fc,fcc), 0.0, 0, -1, ('<s>','<s>'), [])
    for k in range(N):
        if not stacks[k]: continue
        hyps = sorted(stacks[k].values(), key=lambda h: h[0], reverse=True)[:beam]
        for _, score, mask, lend, ctx, out in hyps:
            for (i1,i2) in opts:
                m = sm[(i1,i2)]
                if mask & m: continue
                if lend >= 0 and dlimit >= 0 and abs(i1-(lend+1)) > dlimit: continue
                new_mask  = mask | m
                new_count = k + (i2-i1)
                d = abs(i1-(lend+1)) if lend >= 0 else i1
                dcost = W['distortion']*d
                final = (new_count == N)
                for opt in opts[(i1,i2)]:
                    lm_c = 0.0; nctx = ctx
                    for w in opt['en']: lm_c += lm(w,nctx[0],nctx[1]); nctx=(nctx[1],w)
                    if final: lm_c += lm('</s>',nctx[0],nctx[1])
                    ns = score + static_score(opt,W) + W['lm']*lm_c + dcost
                    rem = 0.0 if final else fc_mask(new_mask,N,fc,fcc)
                    est = ns + rem
                    nle = i2-1
                    key = (new_mask, nle, nctx)
                    prev = stacks[new_count].get(key)
                    if prev is None or est > prev[0]:
                        stacks[new_count][key] = (est, ns, new_mask, nle, nctx, out+list(opt['en']))
    if stacks[N]: return max(stacks[N].values(), key=lambda h: h[1])[5]
    for k in range(N-1,-1,-1):
        if stacks[k]: return max(stacks[k].values(), key=lambda h: h[1])[5]
    return []

print('All SMT functions defined.')

In [ ]:
# ── Alignment helper: write parallel file, run fast_align + atools ────────────
import subprocess

FAST_ALIGN = os.environ['FAST_ALIGN']
ATOOLS     = os.environ['ATOOLS']

def run_alignments(train_fr_raw, train_en_raw, tag):
    """Write parallel file, run fast_align in both directions, symmetrize with gdfa.
    Returns list of alignment link sets (one per sentence)."""
    par_path = f'train_{tag}.parallel'
    fwd_path = f'train_{tag}.fwd'
    rev_path = f'train_{tag}.rev'
    gdfa_path = f'train_{tag}.gdfa'

    # Write fr ||| en parallel file (fast_align expects lowercase, one pair per line).
    with open(par_path, 'w', encoding='utf-8') as fp:
        for f, e in zip(train_fr_raw, train_en_raw):
            fp.write(f.lower() + ' ||| ' + e.lower() + '\n')

    # Reverse parallel file (en ||| fr) for the reverse direction.
    rev_par = f'train_{tag}.rev_parallel'
    with open(rev_par, 'w', encoding='utf-8') as fp:
        for f, e in zip(train_fr_raw, train_en_raw):
            fp.write(e.lower() + ' ||| ' + f.lower() + '\n')

    print(f'  Running fast_align forward ({len(train_fr_raw):,} pairs)...')
    subprocess.run([FAST_ALIGN, '-i', par_path, '-d', '-o', '-v'],
                   stdout=open(fwd_path, 'w'), stderr=subprocess.DEVNULL, check=True)

    print(f'  Running fast_align reverse...')
    subprocess.run([FAST_ALIGN, '-i', rev_par, '-d', '-o', '-v', '-r'],
                   stdout=open(rev_path, 'w'), stderr=subprocess.DEVNULL, check=True)

    print(f'  Symmetrizing with grow-diag-final-and...')
    subprocess.run([ATOOLS, '-i', fwd_path, '-j', rev_path, '-c', 'grow-diag-final-and'],
                   stdout=open(gdfa_path, 'w'), stderr=subprocess.DEVNULL, check=True)

    with open(gdfa_path, encoding='utf-8') as fp:
        alignments = [parse_pharaoh(line) for line in fp]
    return alignments

print('Alignment helpers defined.')

In [ ]:
# ── Main scaling loop ─────────────────────────────────────────────────────────
# Runs both IBM Model 1 and phrase-based SMT at each training size.
# Expects ~2 hours total in Colab (most time is fast_align on 500K pairs).

TRAINING_SIZES = [50_000, 100_000, 250_000, 500_000]
results = []   # list of dicts: {size, ibm_bleu, pb_bleu}

for TRAIN_SIZE in TRAINING_SIZES:
    print(f'\n{"="*60}')
    print(f'Training size: {TRAIN_SIZE:,}')
    print(f'{"="*60}')

    train_idx   = [i for i in range(n) if i not in test_idx_set][:TRAIN_SIZE]
    tr_fr_tok   = [tokenize(fr_raw[i]) for i in train_idx]
    tr_en_tok   = [tokenize(en_raw[i]) for i in train_idx]
    tr_fr_raw   = [fr_raw[i]           for i in train_idx]
    tr_en_raw   = [en_raw[i]           for i in train_idx]

    # ── IBM Model 1 ──
    print('\n[IBM Model 1]')
    t0 = time.time()
    best = train_ibm1(tr_fr_tok, tr_en_tok)
    ibm_score = ibm_bleu(best, test_fr_tok, test_en_ref)
    print(f'  BLEU: {ibm_score:.2f}  ({time.time()-t0:.0f}s)')

    # ── Phrase-based SMT ──
    print('\n[Phrase-based SMT]')
    tag = str(TRAIN_SIZE // 1000) + 'k'
    t0  = time.time()

    print('  Building alignments...')
    aligns = run_alignments(tr_fr_raw, tr_en_raw, tag)

    print('  Building lex tables...')
    pw_ef, pw_fe = build_lex_tables(tr_fr_tok, tr_en_tok, aligns)
    fallback = {f: sorted(d.items(), key=lambda x: x[1], reverse=True)[0][0]
                for f, d in pw_ef.items() if f != NULL_TOKEN}

    print('  Building phrase table...')
    ptable = build_phrase_table(tr_fr_tok, tr_en_tok, aligns, pw_ef, pw_fe)
    print(f'  Phrase table: {len(ptable):,} source entries')

    print('  Building language model...')
    lm = build_lm(tr_en_tok)

    print(f'  Decoding {len(test_fr_tok):,} test sentences...')
    pb_hyps = []
    for idx, fr_s in enumerate(test_fr_tok):
        pb_hyps.append(' '.join(phrase_decode(fr_s, ptable, fallback, pw_ef, lm, W)))
        if (idx+1) % 200 == 0:
            print(f'    {idx+1}/{len(test_fr_tok)}')

    pb_score = sacrebleu.corpus_bleu(pb_hyps, [test_en_ref], lowercase=True).score
    print(f'  BLEU: {pb_score:.2f}  ({time.time()-t0:.0f}s)')

    results.append({'size': TRAIN_SIZE, 'ibm_bleu': ibm_score, 'pb_bleu': pb_score})

print('\n\nAll experiments done.')

In [ ]:
# ── Print results table ───────────────────────────────────────────────────────
print(f'\n{"Training size":>15}  {"IBM BLEU":>10}  {"PB BLEU":>10}')
print('-' * 40)
for r in results:
    print(f'{r["size"]:>15,}  {r["ibm_bleu"]:>10.2f}  {r["pb_bleu"]:>10.2f}')